In [68]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader



In [69]:

dir_loader = DirectoryLoader(
    path="./pdf_documents",  # current directory (AGENTICRAG)
    glob="Residence_Certificate*.pdf",  # only your two PDFs
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_documents = dir_loader.load()

pdf_documents


100%|██████████| 2/2 [00:00<00:00, 203.49it/s]


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-01-25T06:57:18+00:00', 'source': 'pdf_documents/Residence_Certificate_Full_Structured_Info.pdf', 'file_path': 'pdf_documents/Residence_Certificate_Full_Structured_Info.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-01-25T06:57:18+00:00', 'trapped': '', 'modDate': "D:20260125065718+00'00'", 'creationDate': "D:20260125065718+00'00'", 'page': 0}, page_content='Residence Certificate – Structured Information\nService Details\nService ID\n75\nDepartment\nRevenue Administration\nService Name\nResidence Certificate\nAccess Type\nOperator\nOnline Availability\nYes\nService Charge (INR)\n60\nThis document contains cleanly structured and standardized information related to the Residence\nCertificate service under Revenue Administration. The tabular service details a

In [70]:
import sys
print(sys.executable)


/home/ec2-user/ORCHESTRA/.venv/bin/python


In [71]:
# for doc in chunks:
#     source = doc.metadata.get("source", "").lower()

#     if "reason" in source or "explanation" in source:
#         doc.metadata["intent"] = "reasoning"
#     else:
#         doc.metadata["intent"] = "rules"

#     doc.metadata["service"] = "residence_certificate"


In [72]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_files = {
    "rules": "pdf_documents/Residence_Certificate_Full_Structured_Info.pdf",
    "reasoning": "pdf_documents/Residence_Certificate_Service_Explanation_and_Document_Reasons.pdf"
}


documents = {}

for key, path in pdf_files.items():
    loader = PyMuPDFLoader(path)
    documents[key] = loader.load()


In [73]:
from langchain_core.documents import Document

def make_chunk(text, intent, source, entity=None):
    metadata = {
        "service": "residence_certificate",
        "intent": intent,
        "source": source
    }
    if entity:
        metadata["entity"] = entity

    return Document(page_content=text.strip(), metadata=metadata)



In [74]:
#Semantic chunking for PDF 1

rule_chunks = []

rules_text = "\n".join([d.page_content for d in documents["rules"]])

# --- Service Info ---
service_info = rules_text.split("Mandatory Documents")[0]
rule_chunks.append(
    make_chunk(service_info, "service_info", pdf_files["rules"])
)

# --- Mandatory Documents ---
mandatory_section = rules_text.split("Mandatory Documents")[1].split("Category-wise")[0]
rule_chunks.append(
    make_chunk(mandatory_section, "rules_mandatory", pdf_files["rules"])
)

# --- Category-wise Sections ---
citizen_section = rules_text.split("General Citizens")[1].split("Government employees")[0]
rule_chunks.append(
    make_chunk(citizen_section, "rules_category_citizen", pdf_files["rules"])
)

govt_section = rules_text.split("Government employees")[1]
rule_chunks.append(
    make_chunk(govt_section, "rules_category_govt", pdf_files["rules"])
)


In [75]:
#Semantic chunking for PDF 2

reasoning_chunks = []

reasoning_text = "\n".join([d.page_content for d in documents["reasoning"]])

# --- Service explanation ---
service_explanation = reasoning_text.split("Why Documents are Collected")[0]
reasoning_chunks.append(
    make_chunk(service_explanation, "service_explanation", pdf_files["reasoning"])
)

# --- Global reasoning ---
global_reason = reasoning_text.split("Why Documents are Collected")[1].split("Reasoning Behind")[0]
reasoning_chunks.append(
    make_chunk(global_reason, "reasoning_global", pdf_files["reasoning"])
)

# --- Per-document reasoning ---
document_reasons = {
    "Applicant Photograph": "Applicant Photograph",
    "Current Address Proof": "Current Address Proof",
    "Self-Declaration": "Self-Declaration",
    "Passport": "Passport",
    "Driving Licence": "Driving Licence",
    "PAN Card": "PAN Card",
    "Bank / Post Office Passbook": "Bank / Post Office Passbook",
    "Smart Card": "Smart Card",
    "Health Insurance Smart Card": "Health Insurance Smart Card",
    "Pension Document": "Pension Document",
    "Service Identity Card": "Service Identity Card",
    "MP/MLA/MLC Identity Card": "MP/MLA/MLC Identity Card",
    "Photo Voter Slip": "Photo Voter Slip"
}

for doc_name, key in document_reasons.items():
    if doc_name in reasoning_text:
        section = reasoning_text.split(doc_name)[1].split("\n", 1)[1]
        reasoning_chunks.append(
            make_chunk(section, "reasoning_document", pdf_files["reasoning"], entity=doc_name)
        )


In [76]:
all_chunks = rule_chunks + reasoning_chunks

print(f"Total semantic chunks created: {len(all_chunks)}")

# Inspect one
print(all_chunks[0].metadata)
print(all_chunks[0].page_content[:300])


Total semantic chunks created: 19
{'service': 'residence_certificate', 'intent': 'service_info', 'source': 'pdf_documents/Residence_Certificate_Full_Structured_Info.pdf'}
Residence Certificate – Structured Information
Service Details
Service ID
75
Department
Revenue Administration
Service Name
Residence Certificate
Access Type
Operator
Online Availability
Yes
Service Charge (INR)
60
This document contains cleanly structured and standardized information related to the


In [77]:
from dotenv import load_dotenv
import os

load_dotenv()  # 👈 this reads .env into environment variables

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
NVIDIA_LLM_API_KEY = os.getenv("NVIDIA_LLM_API_KEY")   

if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found")



In [78]:
from openai import OpenAI

client_embed = OpenAI(
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1"
)


In [79]:
def embed_chunks(chunks):
    """
    chunks: List[langchain_core.documents.Document]
    returns: List[List[float]] -> embeddings aligned with chunks
    """

    texts = [chunk.page_content for chunk in chunks]

    response = client_embed.embeddings.create(
        model="nvidia/nv-embedqa-e5-v5",
        input=texts,
        extra_body={
            "input_type": "passage"  # REQUIRED for document chunks
        }
    )

    return [item.embedding for item in response.data]


In [80]:
# %% 
chunk_embeddings = embed_chunks(all_chunks)

# sanity check
print(len(chunk_embeddings))       # should be 19
print(len(chunk_embeddings[0]))    # embedding dimension (~1024)


19
1024


In [81]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url="https://103795bc-13b7-45b8-aea6-9b2ab07095a1.eu-west-2-0.aws.cloud.qdrant.io", 
    api_key=QDRANT_API_KEY,
)



In [82]:
from qdrant_client.models import VectorParams, Distance

VECTOR_SIZE = len(chunk_embeddings[0])

COLLECTION_NAME = "residence_certificate_agent"

qdrant_client.recreate_collection(
    collection_name="residence_certificate_agent",
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE   # ✅ BEST for E5 models
    )
)


/tmp/ipykernel_2992/511622667.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


True

In [83]:
# %%
from qdrant_client.models import PayloadSchemaType

qdrant_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="intent",
    field_schema=PayloadSchemaType.KEYWORD
)

print("Index created for 'intent'")


Index created for 'intent'


In [84]:
# %%
qdrant_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="entity",
    field_schema=PayloadSchemaType.KEYWORD
)

print("Index created for 'entity'")


Index created for 'entity'


In [85]:
# %%
qdrant_client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="service",
    field_schema=PayloadSchemaType.KEYWORD
)

print("Index created for 'service'")


Index created for 'service'


In [86]:
# %%
from qdrant_client.models import PointStruct
import uuid

points = []

for chunk, embedding in zip(all_chunks, chunk_embeddings):
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=embedding,
            payload={
                "text": chunk.page_content,
                **chunk.metadata
            }
        )
    )

qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print(f"Inserted {len(points)} chunks into Qdrant")


Inserted 19 chunks into Qdrant


In [87]:
# %%
info = qdrant_client.get_collection(COLLECTION_NAME)
print(info)


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=19 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_retai

In [88]:
# %%
def embed_query(text: str):
    response = client_embed.embeddings.create(
        model="nvidia/nv-embedqa-e5-v5",
        input=text,
        extra_body={
            "input_type": "query"   # 🔑 REQUIRED for queries
        }
    )
    return response.data[0].embedding


In [89]:
# %%
from qdrant_client.models import Filter, FieldCondition, MatchValue

def search_mandatory_documents(question: str, limit=5):
    query_vector = embed_query(question)

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[],
        query=query_vector,
        limit=limit,
        query_filter=Filter(
            must=[
                FieldCondition(
                    key="intent",
                    match=MatchValue(value="rules_mandatory")
                )
            ]
        )
    )

    return results.points


In [90]:
# %%
results = search_mandatory_documents(
    "What are the documents to be certified by Government employees ?"
)

for hit in results:
    print("Score:", hit.score)
    print(hit.payload["text"][:300])
    print("-" * 50)


Score: 0.31814763
(Required for All Applicants)
The following documents must be submitted by every applicant without exception:
1. Applicant Photograph
2. Current Address Proof
3. Self-Declaration of Applicant
--------------------------------------------------


In [91]:
def build_context(results):
    """
    results: list of Qdrant points
    returns: single context string for LLM
    """
    context_blocks = []
    for hit in results:
        context_blocks.append(hit.payload["text"])

    return "\n\n".join(context_blocks)


In [92]:
from openai import OpenAI


client_embed_reasoning = OpenAI(
    api_key=NVIDIA_LLM_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1"
)


In [96]:
def generate_answer(question: str, context: str):
    response = client_embed_reasoning.chat.completions.create(
        model="meta/llama-3.1-8b-instruct",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a government service assistant. "
                    "Answer ONLY using the provided context. "
                    "If the answer is not present in the context, say "
                    "'The information is not available in the provided documents.'"
                )
            },
            {
                "role": "user",
                "content": f"""
Context:
{context}

Question:
{question}
"""
            }
        ],
        temperature=0.2,
        max_tokens=512
    )

    return response.choices[0].message.content


In [97]:
def answer_question(user_query: str):
    results = search_mandatory_documents(user_query)
    context = build_context(results)
    answer = generate_answer(user_query, context)
    return answer


In [101]:
response = answer_question(
    "What are the certificates i have to submit to apply for residence certificate if i am a government employee?"
)

print(response)


The information is not available in the provided documents.
